# ERC : flux d'entrée, suivi hebdomadaire

Le modèle de ligne de démontage et de tri part de ce flux et n'est fiable qu'à la granularité où il est vérifié.

| # | Tableau | Sert à |
|---|---|---|
| 1 | Composant, FRU, type de déchet, benne | savoir ce que contient chaque flux |
| 2 | FRU rebutées par semaine et par type | le flux d'entrée du modèle ($N_{ft}$) et sa variabilité |
| 3 | Masse estimée par semaine, avec marge d'erreur | la valeur estimée |
| 4 | Estimé contre pesé : écart, cause de déviation | la valeur mesurée et la confiance dans l'estimation |
| 5 | Verdict par flux | ce que le modèle a le droit d'utiliser |

À exécuter après `ERC_Aide_Decision_Composants.ipynb`, dans le même dossier, et avant `ERC_Fiabilite_Risques_Priorisation.ipynb` qui relit ses sorties. Pesées : `Acquisition/pesees_hebdo.csv` (séparateur `;`), colonnes `Semaine;Flux;Masse mesurée (kg)` et, facultatives, `Incertitude mesure (kg);Périmètre;Commentaire`. `Flux` reprend le nom de la benne (ou du type de déchet, selon `PAR`).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJET = Path.cwd()
sys.path.insert(0, str(PROJET))
import erc_flux_hebdo as efh

F_AIDE = PROJET / "ERC_Aide_Decision_Composants.xlsx"
F_VALO = PROJET.parent / "ERC_Valorisation_MICV_v1.xlsx"          # poids des FRU (QueryERC_Flow_In_Out, Net weight)
F_PESEES = PROJET / "Acquisition" / "pesees_hebdo.csv"
F_SORTIE = PROJET / "ERC_Flux_Entree_Suivi_Hebdo.xlsx"
PAR = "Benne"                      # ou "Type de déchet"
COL_DATE_REBUT = None              # None = recherche automatique, la règle retenue est affichée ; sinon nom exact de la colonne
efh.INCERTITUDE_POIDS = None       # ex. 0.10 quand la dispersion des poids unitaires est mesurée
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 220)

cl = pd.read_excel(F_AIDE, sheet_name="Classement_Tous_ateliers")
ev, couverts = efh.charger_evenements(PROJET)
print(f"{len(ev)} lignes de consommation, {ev['Qte'].sum():.0f} pièces, {len(couverts)} jours extraits")

## 1. Tableau composant, FRU, type de déchet

Une ligne par couple (FRU, composant) observé dans les extractions : sur quel FRU la pièce est consommée, son type de déchet, sa benne, son poids. Type de déchet ou benne vide = pièce pas encore classée ; poids vide = pièce à peser.

In [ ]:
comp = efh.tableau_composants(cl)                       # une ligne par composant (sert à l'estimation)
comp_fru = efh.tableau_composants(cl, ev)               # une ligne par couple FRU x composant
synthese = comp.groupby(["Benne", "Type de déchet"], dropna=False).agg(
    Références=("Référence", "nunique"), Pièces=("Pièces", "sum"), **{"Références au poids connu": ("Poids connu", "sum"),
    "Kg (poids connus)": ("Kg (poids connus)", "sum")}).reset_index()
print(f"{comp_fru['FRU'].nunique()} FRU, {comp_fru['Référence'].nunique()} composants, {len(comp_fru)} couples | "
      f"sans type de déchet : {int(comp_fru['Type de déchet'].isna().sum())} | sans poids : {int(comp_fru['Poids unitaire (kg)'].isna().sum())}")
display(synthese)
comp_fru.head(40)

## 2. FRU rebutées par semaine : le flux d'entrée du modèle

Une semaine extraite sans rebut compte pour **0**, elle n'est pas absente : c'est ce qui mesure la variabilité du flux. `rebut_an` (bas, moyen, haut) est calculé sur les semaines complètes par ré-échantillonnage ; il alimente directement le modèle de ligne. La date de rebut utilisée est affichée : à confirmer avec le responsable flux.

In [ ]:
fr, regle_date = efh.charger_fru_rebutees(PROJET, COL_DATE_REBUT)
print("Date de rebut :", regle_date, "|", len(fr), "lignes")
poids_fru = None
if F_VALO.is_file():
    fx = pd.read_excel(F_VALO, sheet_name="QueryERC_Flow_In_Out")
    poids_fru = pd.to_numeric(fx["Net weight"], errors="coerce").groupby(fx["ITEM"].map(efh.ref)).first()
    poids_fru = poids_fru[poids_fru.gt(0)]
flux_fru = efh.flux_fru_hebdo(fr, couverts, poids_fru) if len(fr) else pd.DataFrame(columns=["Semaine", "FRU", "FRU rebutées", "Jours extraits", "Semaine complète"])
rebut = efh.rebut_annuel(flux_fru) if len(flux_fru) else pd.DataFrame(columns=["FRU", "rebut_an"])
if len(flux_fru):
    display(flux_fru.pivot_table(index="Semaine", columns="FRU", values="FRU rebutées", aggfunc="sum").assign(Total=lambda d: d.sum(axis=1)))
rebut.drop(columns="rebut_an", errors="ignore").round(1)

## 3. Masse estimée par semaine

Borne basse : pièces au poids connu seulement. Borne haute : les pièces sans poids supposées au poids moyen de leur flux. La marge d'erreur est la demi-largeur de cet intervalle ; elle se resserre à mesure que les poids manquants sont pesés.

In [ ]:
est = efh.estimation_hebdo(ev, comp, couverts, par=PAR)
est.round(2)

## 4. Semaine, valeur estimée et mesurée, marge d'erreur, cause de déviation

Seules les semaines qui ont une pesée sont comparées. « Dans la marge » tient compte de l'incertitude de la balance. Le cumul par flux absorbe le décalage des bacs individuels : si l'écart cumulé est nettement plus faible que l'écart de la semaine, c'est un décalage dans le temps et non une erreur de niveau. La dernière colonne est à remplir avec la cause constatée sur place.

In [ ]:
if F_PESEES.is_file():
    pesees = pd.read_csv(F_PESEES, sep=";", encoding="utf-8-sig", decimal=",")
    rap = efh.rapprocher(est, pesees)
else:
    print("Pas encore de fichier de pesées :", F_PESEES)
    rap = efh.rapprocher(est, pd.DataFrame(columns=["Semaine", "Flux", "Masse mesurée (kg)"]))
vue = ["Semaine", "Flux", "Jours extraits", "Estimé bas (kg)", "Estimé central (kg)", "Estimé haut (kg)", "Marge d'erreur (± kg)", "Masse mesurée (kg)",
       "Écart (kg)", "Écart (%)", "Dans la marge", "Écart cumulé (%)", "Causes candidates de l'écart", "Cause constatée (à saisir)"]
rap[vue].round(2)

## 5. Verdict par flux

In [ ]:
comparables = rap[rap["Dans la marge"].ne("non comparable")]
verdict = comparables.groupby("Flux").agg(**{"Semaines comparées": ("Semaine", "nunique"), "Dans la marge": ("Dans la marge", lambda s: (s == "oui").sum()),
                                           "Écart moyen (%)": ("Écart (%)", "mean"), "Écart cumulé final (%)": ("Écart cumulé (%)", "last"),
                                           "Couverture poids moyenne": ("Couverture poids", "mean")}).reset_index()
verdict["Utilisable par le modèle"] = np.where((verdict["Semaines comparées"] >= 3) & (verdict["Écart cumulé final (%)"].abs() <= 0.15), "oui",
                                              np.where(verdict["Semaines comparées"] < 3, "pas assez de semaines (3 minimum)", "non : corriger la cause dominante"))
display(verdict.round(3))
rebut_x = rebut.assign(rebut_an=rebut["rebut_an"].map(repr)) if len(rebut) else rebut
with pd.ExcelWriter(F_SORTIE) as w:
    for n, d in {"Composant_FRU_type_dechet": comp_fru, "Composants_type_dechet": comp, "Synthese_bennes": synthese, "Flux_FRU_hebdo": flux_fru,
                 "Rebut_annuel": rebut_x, "Estime_hebdo": est, "Rapprochement": rap, "Verdict": verdict}.items():
        d.to_excel(w, sheet_name=n, index=False)
print("Écrit :", F_SORTIE.name)